In [0]:
#Leitura camada bronze
df_cartao = spark.table("bronze.cartao_diaria_05hs")

In [0]:
from pyspark.sql import functions as F

df_silver_cartao = (
  df_cartao.select(
    F.col("id").cast("long").alias("id"),
    F.col("id_conta").cast("long").alias("id_conta"),
    F.trim(F.col("num_cartao").cast("string")).alias("num_cartao"),
    F.initcap(F.trim(F.col("nom_impresso"))).alias("nom_impresso"),
    F.col("data_criacao_cartao").cast("date").alias("data_criacao_cartao"),
    F.current_timestamp().alias("dt_ingestao_silver")
  )
  .dropDuplicates(["id"])
)


In [0]:
# Qualidade dos dados
erros = []  # Lista para armazenar erros de qualidade identificados

# Verifica se num_cartao é nulo ou inválido (menos de 16 caracteres)
if df_silver_cartao.filter((F.col("num_cartao").isNull()) | (F.length("num_cartao") < 16)).count() > 0:
    erros.append("num_cartao nulo/inválido")

# Verifica se id é nulo
if df_silver_cartao.filter(F.col("id").isNull()).count() > 0:
    erros.append("id nulo")

# Verifica se id_conta é nulo
if df_silver_cartao.filter(F.col("id_conta").isNull()).count() > 0:
    erros.append("id_conta nulo")

# Verifica se nom_impresso é nulo ou vazio
if df_silver_cartao.filter((F.col("nom_impresso").isNull()) | (F.col("nom_impresso") == "")).count() > 0:
    erros.append("nom_impresso nulo/vazio")

# Verifica se data_criacao_cartao é nulo
if df_silver_cartao.filter(F.col("data_criacao_cartao").isNull()).count() > 0:
    erros.append("data_criacao_cartao nulo")

# Se houver erros, imprime e lança exceção caso necessario
if erros:
    for erro in erros:
        print(f"Falha qualidade cartao_silver: {erro}")
    raise Exception("Falha qualidade cartao_silver: " + " | ".join(erros))

In [0]:
#Salvando base na camada silver e modo overwrite
(df_silver_cartao.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("silver.cartao_diaria_05hs"))